# edge-of-the-world — getting started

`bw_eotw` (edge-of-the-world) is an alternate Brightway database backend that lets a single edge definition expand into **multiple matrix cells**, selected by a *config* dict at `process()` time.

This notebook walks through:

1. Setting up an `eotw` database
2. The `standard` interpreter (plain bw2data-compatible exchange)
3. The `temporal` interpreter (year-keyed lookup)
4. The `loss` interpreter (main flow + proportional loss)
5. The `scenario` interpreter (strict scenario selection)
6. The `provider_mix` interpreter (splits demand across providers)
7. The `temporal_scenario` interpreter (scenario × year)
8. Writing a custom interpreter

In [ ]:
import bw2data
import bw_eotw  # registers the eotw backend and all built-in interpreters
from bw_eotw import MatrixEntry, RichEdge, register

bw2data.projects.set_current("eotw-getting-started")
bw2data.preferences['biosphere_database'] = 'biosphere'
print("Project:", bw2data.projects.current)

## 1  Setting up an `eotw` database

Pass `backend="eotw"` to `bw2data.Database` to get a `RichEdgesBackend` instead of the default `SQLiteBackend`.  Everything else — writing data, querying nodes, iterating exchanges — works exactly the same way.

In [ ]:
if 'demo' in bw2data.databases:
    del bw2data.databases['demo']

db = bw2data.Database('demo', backend='eotw')
db.register()
print(type(db).__name__)   # RichEdgesBackend

In [ ]:
# Write two simple nodes
db.write({
    ('demo', 'electricity'): {
        'name': 'electricity production',
        'unit': 'kWh',
        'type': 'process',
    },
    ('demo', 'heat'): {
        'name': 'heat production',
        'unit': 'MJ',
        'type': 'process',
    },
})
electricity = db.get('electricity')
heat        = db.get('heat')
print(electricity, heat)

## 2  Standard interpreter

Edges without an `interpreter` key (or with `"interpreter": "standard"`) behave exactly like plain bw2data exchanges.  No config is needed.

In [ ]:
from bw_eotw.registry import resolve

edge_data = {
    'row': electricity.id,
    'col': heat.id,
    'interpreter': 'standard',
    'amount': 0.5,
    'flip': False,
}
entries = list(resolve(edge_data, {}))
print(entries[0])   # MatrixEntry with amount=0.5

## 3  Temporal interpreter

Store a time-series of values in `temporal_values`.  Pass `config={"year": ...}` at `process()` time; an optional per-edge `default_year` is used when the config year is absent or not found.

In [ ]:
temporal_edge = {
    'row': electricity.id,
    'col': heat.id,
    'interpreter': 'temporal',
    'temporal_values': {
        2010: 0.30,
        2020: 0.40,
        2030: 0.60,
    },
    'default_year': 2020,
    'flip': False,
}

for year in [2010, 2020, 2030]:
    entries = list(resolve(temporal_edge, {'year': year}))
    print(f"year={year} → amount={entries[0].amount}")

# Year not in table → falls back to default_year=2020
entries = list(resolve(temporal_edge, {'year': 2025}))
print(f"year=2025 (not in table) → amount={entries[0].amount}  (default_year fallback)")

# No year in config → falls back to default_year=2020
entries = list(resolve(temporal_edge, {}))
print(f"no year in config → amount={entries[0].amount}  (default_year fallback)")

## 4  Loss interpreter

A single edge expands to **two** matrix entries: the net flow and the loss component.  This is useful for transmission losses, process yields, etc.

```
  main entry  = amount
  loss entry  = amount × loss_factor
```

In [ ]:
loss_edge = {
    'row': electricity.id,
    'col': heat.id,
    'interpreter': 'loss',
    'amount': 1.0,
    'loss_factor': 0.05,   # 5% loss
    'flip': False,
}

entries = list(resolve(loss_edge, {}))
print(f"main  entry: amount={entries[0].amount}")   # 1.0
print(f"loss  entry: amount={entries[1].amount}")   # 0.05
print(f"total gross: {entries[0].amount + entries[1].amount}")  # 1.05

`loss_factor` can also be an uncertainty dict to propagate uncertainty into the loss entry:

In [ ]:
loss_edge_uncertain = {
    'row': electricity.id,
    'col': heat.id,
    'interpreter': 'loss',
    'amount': 2.0,
    'loss_factor': {'amount': 0.05, 'uncertainty_type': 2, 'scale': 0.01},
    'flip': False,
}

entries = list(resolve(loss_edge_uncertain, {}))
loss = entries[1]
print(f"loss amount={loss.amount}, scale={loss.scale}, uncertainty_type={loss.uncertainty_type}")

## 5  Scenario interpreter

Store one value per named scenario.  The config **must** supply `"scenario"`; a `KeyError` is raised if it's absent or the name doesn't match.

In [ ]:
scenario_edge = {
    'row': electricity.id,
    'col': heat.id,
    'interpreter': 'scenario',
    'scenario_values': {
        'baseline':   1.00,
        'optimistic': 0.70,
        'pessimistic': 1.40,
    },
    'flip': False,
}

for scenario in ['baseline', 'optimistic', 'pessimistic']:
    entries = list(resolve(scenario_edge, {'scenario': scenario}))
    print(f"scenario={scenario!r:12s} → amount={entries[0].amount}")

# Missing scenario → KeyError
try:
    resolve(scenario_edge, {})
except KeyError as exc:
    print(f"\nMissing scenario key: {exc}")

## 6  Provider-mix interpreter

Splits a product demand across multiple provider nodes by fractional share.  Each provider maps to a separate row in the technosphere matrix.  Shares must sum to 1 (using `math.isclose`).

In [ ]:
if 'grid' not in bw2data.databases:
    grid_db = bw2data.Database('grid', backend='eotw')
    grid_db.register()
    grid_db.write({
        ('grid', 'wind'):  {'name': 'wind power',   'unit': 'kWh', 'type': 'process'},
        ('grid', 'solar'): {'name': 'solar power',  'unit': 'kWh', 'type': 'process'},
        ('grid', 'gas'):   {'name': 'gas power',    'unit': 'kWh', 'type': 'process'},
    })

mix_edge = {
    'row': electricity.id,   # overridden per provider — col is fixed
    'col': heat.id,
    'interpreter': 'provider_mix',
    'product_name': 'electricity',
    'amount': 100.0,          # total demand to split
    'mix': [
        {'input': ('grid', 'wind'),  'share': 0.40},
        {'input': ('grid', 'solar'), 'share': 0.35},
        {'input': ('grid', 'gas'),   'share': 0.25},
    ],
    'flip': False,
}

entries = list(resolve(mix_edge, {}))
for e in entries:
    print(f"row={e.row}, col={e.col}, amount={e.amount}")

## 7  Temporal-scenario interpreter

Combines strict scenario selection with year-keyed lookup.  The outer dict maps scenario names to year dicts; `default_year` is optional.

In [ ]:
ts_edge = {
    'row': electricity.id,
    'col': heat.id,
    'interpreter': 'temporal_scenario',
    'scenario_temporal_values': {
        'baseline': {
            2020: 1.00,
            2030: 0.85,
        },
        'optimistic': {
            2020: 0.80,
            2030: {'amount': 0.60, 'uncertainty_type': 2, 'scale': 0.05},
        },
    },
    'default_year': 2020,
    'flip': False,
}

for scenario in ['baseline', 'optimistic']:
    for year in [2020, 2030]:
        entries = list(resolve(ts_edge, {'scenario': scenario, 'year': year}))
        e = entries[0]
        print(f"scenario={scenario!r:12s}, year={year} → amount={e.amount}")

# Year not in optimistic → falls back to default_year=2020
entries = list(resolve(ts_edge, {'scenario': 'optimistic', 'year': 2025}))
print(f"\noptimistic, year=2025 (missing) → amount={entries[0].amount}  (default_year fallback)")

## 8  Writing a custom interpreter

Use the `@register` decorator to add your own interpreter.  The function receives the full edge data dict and the config dict, and should `yield` one or more `MatrixEntry` instances.

Use `@register_validator` to validate edge data at save time.

In [ ]:
from collections.abc import Iterator
from bw_eotw import register, register_validator, MatrixEntry

@register('scaled')
def scaled_interpreter(edge_data: dict, config: dict) -> Iterator[MatrixEntry]:
    """Multiply the stored amount by a scale_factor from the config."""
    scale = config.get('scale_factor', 1.0)
    amount = edge_data['amount'] * scale
    yield MatrixEntry(
        row=edge_data['row'],
        col=edge_data['col'],
        amount=amount,
        flip=edge_data.get('flip', False),
    )

@register_validator('scaled')
def validate_scaled(edge_data: dict) -> None:
    if 'amount' not in edge_data:
        raise ValueError("scaled edges must have 'amount'")

# Try it out
custom_edge = {
    'row': electricity.id,
    'col': heat.id,
    'interpreter': 'scaled',
    'amount': 10.0,
    'flip': False,
}

from bw_eotw.registry import resolve

for factor in [0.5, 1.0, 2.0]:
    entries = list(resolve(custom_edge, {'scale_factor': factor}))
    print(f"scale_factor={factor} → amount={entries[0].amount}")

## 9  Using `db.process()` with a config

The config dict is passed to `process()` on the backend.  Every interpreter for every edge receives the same config.

In [ ]:
# Process with year=2030 and scenario='optimistic'
# (edges that don't use those keys simply ignore them)
# db.process(config={'year': 2030, 'scenario': 'optimistic'})
print("Call db.process(config={'year': 2030, 'scenario': 'optimistic'}) on a populated database")
print("to generate the bw_processing datapackage with resolved matrix values.")

## 10  Validation at save time

`RichEdge.save()` calls `validate_edge()` before writing to the database, so bad edge data is caught early.

In [ ]:
from bw_eotw.registry import validate_edge

# Loss factor outside [0, 1]
try:
    validate_edge({'interpreter': 'loss', 'loss_factor': 1.5})
except ValueError as exc:
    print(f"loss_factor=1.5: {exc}")

# Provider-mix shares not summing to 1
try:
    validate_edge({
        'interpreter': 'provider_mix',
        'product_name': 'electricity',
        'mix': [
            {'input': ('db', 'a'), 'share': 0.4},
            {'input': ('db', 'b'), 'share': 0.4},   # total = 0.8, not 1
        ],
    })
except ValueError as exc:
    print(f"bad shares: {exc}")

# temporal_scenario with default_year missing from one scenario
try:
    validate_edge({
        'interpreter': 'temporal_scenario',
        'scenario_temporal_values': {
            'baseline':   {2020: 1.0},
            'optimistic': {2030: 0.7},   # 2020 absent here
        },
        'default_year': 2020,
    })
except ValueError as exc:
    print(f"default_year missing: {exc}")

In [ ]:
# Clean up
del bw2data.databases['demo']
if 'grid' in bw2data.databases:
    del bw2data.databases['grid']